# C11-neural-training — Practice p24 — Solution


**Type:** challenge · **Difficulty:** advanced · **Concepts:** torch-optimizers, trained-mlp, batch-normalization, dropout


Each configuration begins from the same seeded construction point, snapshots
parameters and BatchNorm buffers, and lets Adam own only trainable parameters.
Mode probes are computed from the trained model rather than hard-coded flags.


In [ ]:
import torch
import torch.nn as nn

def run_ablation(seed=20260804,epochs=250):
    torch.use_deterministic_algorithms(True); torch.set_default_dtype(torch.float64)
    g=torch.Generator(device="cpu").manual_seed(seed); centers=torch.tensor([[-2.,-1.],[0.,2.],[2.,-1.]],dtype=torch.float64)
    X=torch.cat([center+.30*torch.randn((40,2),generator=g,dtype=torch.float64) for center in centers]); y=torch.arange(3).repeat_interleave(40)
    configs={"full":(True,.25,False),"frozen_affine":(True,.25,True),"no_dropout":(True,0.,False),"no_batchnorm":(False,.25,False)}
    output={}
    for name,(use_bn,p_drop,freeze) in configs.items():
        torch.manual_seed(seed)
        layers=[nn.Linear(2,12)]
        if use_bn: layers.append(nn.BatchNorm1d(12))
        layers.extend([nn.ReLU(),nn.Dropout(p_drop),nn.Linear(12,3)])
        model=nn.Sequential(*layers).to(dtype=torch.float64,device="cpu"); bn=next((m for m in model if isinstance(m,nn.BatchNorm1d)),None)
        if freeze: bn.weight.requires_grad_(False); bn.bias.requires_grad_(False)
        initial={n:p.detach().clone() for n,p in model.named_parameters()}
        buffers=None if bn is None else (bn.running_mean.clone(),bn.running_var.clone())
        owned=[p for p in model.parameters() if p.requires_grad]; optimizer=torch.optim.Adam(owned,lr=.03); criterion=nn.CrossEntropyLoss()
        losses=torch.empty(epochs,dtype=torch.float64); model.train()
        for epoch in range(epochs):
            optimizer.zero_grad(set_to_none=True); loss=criterion(model(X),y); losses[epoch]=loss.detach(); loss.backward(); optimizer.step()
        linear_names=[n for n,_ in model.named_parameters() if ".weight" in n or ".bias" in n]
        linear_names=[n for n in linear_names if not (bn is not None and n.startswith(str(list(model).index(bn))+"."))]
        current=dict(model.named_parameters()); linear_movement=max(float(torch.linalg.vector_norm(current[n].detach()-initial[n])) for n in linear_names)
        if bn is None: affine_movement=None; buffer_changed=None
        else:
            prefix=str(list(model).index(bn)); affine_movement=max(float(torch.linalg.vector_norm(current[f"{prefix}.{s}"].detach()-initial[f"{prefix}.{s}"])) for s in ("weight","bias"))
            buffer_changed=not (torch.equal(buffers[0],bn.running_mean) and torch.equal(buffers[1],bn.running_var))
        model.eval()
        with torch.no_grad(): accuracy=float((model(X).argmax(1)==y).double().mean())
        model.train(); torch.manual_seed(seed); train_a=model(X).detach(); train_b=model(X).detach(); train_equal=bool(torch.allclose(train_a,train_b,atol=1e-9,rtol=1e-7))
        model.eval()
        with torch.no_grad(): eval_a=model(X); eval_b=model(X)
        eval_equal=bool(torch.allclose(eval_a,eval_b,atol=1e-9,rtol=1e-7))
        output[name]={"initial_loss":float(losses[0]),"final_loss":float(losses[-1]),"accuracy":accuracy,
            "linear_movement":linear_movement,"bn_affine_movement":affine_movement,"bn_buffer_changed":buffer_changed,
            "optimizer_state_entries":len(optimizer.state),"train_repeat_equal":train_equal,"eval_repeat_equal":eval_equal,"model":model}
        model._training_optimizer=optimizer
    return output

X_probe_p24=torch.tensor([[-2.,-1.],[0.,2.],[2.,-1.]],dtype=torch.float64).repeat((40,1))
result_p24=run_ablation()


### Answer check


In [ ]:
assert set(result_p24)=={"full","frozen_affine","no_dropout","no_batchnorm"}
for name,row in result_p24.items():
    model=row["model"]; optimizer=model._training_optimizer
    assert row["final_loss"]<row["initial_loss"] and row["linear_movement"]>.05
    owned=[p for p in model.parameters() if p.requires_grad]
    assert row["optimizer_state_entries"]==len(owned)==len(optimizer.state)
    assert {id(p) for group in optimizer.param_groups for p in group["params"]}=={id(p) for p in owned}
    drop=next(module for module in model.modules() if isinstance(module,nn.Dropout))
    bn=next((module for module in model.modules() if isinstance(module,nn.BatchNorm1d)),None)
    model.train(); torch.manual_seed(9917)
    probe_a=model(X_probe_p24).detach(); probe_b=model(X_probe_p24).detach()
    observed_train_equal=bool(torch.allclose(probe_a,probe_b,atol=1e-9,rtol=1e-7))
    model.eval(); buffers_before=None if bn is None else (bn.running_mean.clone(),bn.running_var.clone())
    with torch.no_grad(): eval_a=model(X_probe_p24); eval_b=model(X_probe_p24)
    buffers_after=None if bn is None else (bn.running_mean.clone(),bn.running_var.clone())
    assert bool(torch.allclose(eval_a,eval_b,atol=1e-9,rtol=1e-7)) and row["eval_repeat_equal"]
    assert observed_train_equal==row["train_repeat_equal"]==(drop.p==0.0)
    assert torch.allclose(drop(torch.ones(5,12)),torch.ones(5,12),atol=1e-9,rtol=1e-7)
    if bn is not None: assert all(torch.equal(a,b) for a,b in zip(buffers_before,buffers_after))
assert result_p24["full"]["accuracy"]>=.95
assert result_p24["frozen_affine"]["bn_affine_movement"]==0.0 and result_p24["full"]["bn_affine_movement"]>0.0
assert result_p24["full"]["bn_buffer_changed"] and result_p24["frozen_affine"]["bn_buffer_changed"] and result_p24["no_dropout"]["bn_buffer_changed"]
assert result_p24["no_batchnorm"]["bn_affine_movement"] is None and result_p24["no_batchnorm"]["bn_buffer_changed"] is None
